
# TOPS Rosseland Mean Opacity: $\kappa(T,\rho)$ for Solar Composition

The TOPS opacity code (Los Alamos National Laboratory) provides tables of the
Rosseland mean opacity $\kappa_R(T,\rho)$ for arbitrary mixtures of
elements.  Triceratops ships a bundled table for solar composition
($X \approx 0.70,\;Z \approx 0.02$) from which opacity and its
log-space derivatives are evaluated via bilinear interpolation.

Unlike the OPAL tables — which are parameterised on a
$(\log_{10}T,\,\log_{10}R)$ grid where
$R \equiv \rho\,T_6^{-3}$ — the TOPS table is stored on a direct
$(\log_{10}T,\,\log_{10}\rho)$ grid, so a fixed density corresponds
to a fixed grid column at all temperatures.

This example performs the canonical sanity check: plotting
$\kappa_R$ as a function of temperature for several densities,
reproducing the characteristic features of stellar-interior opacity:

- The **electron-scattering plateau** at high $T$ ($\kappa \approx 0.34\;\mathrm{cm^2\,g^{-1}}$),
- The **Kramers rise** at intermediate temperatures ($\kappa \propto \rho\,T^{-3.5}$),
- The **opacity peak** near $T \sim 10^5\;\mathrm{K}$ driven by iron-group line opacity,
- The rapid **drop at low** $T$ as hydrogen and helium become neutral.

<div class="alert alert-info"><h4>Note</h4><p>The bundled TOPS table covers
   $5\times10^{-4}\;\mathrm{keV} \lesssim T \lesssim 100\;\mathrm{keV}$
   ($\approx 5.8\times10^3\;\mathrm{K}$ to $1.16\times10^9\;\mathrm{K}$)
   and $10^{-7} \le \rho\,[\mathrm{g\,cm^{-3}}] \le 10^2$.
   Queries outside this domain — or at high-density / low-temperature corners
   that TOPS clamped during table generation — are returned as ``NaN`` and
   appear as gaps.</p></div>

## Relevant API references
- :func:`~triceratops.radiation.opacity.utils.load_tops_opacity`
- :class:`~triceratops.radiation.opacity.grey_opacity.tops.TOPSOpacity`


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

from triceratops.radiation.opacity import load_tops_opacity
from triceratops.utils.plot_utils import set_plot_style

## Load the Default Solar Opacity

:func:`~triceratops.radiation.opacity.utils.load_tops_opacity` loads the
bundled ``tops_solar.dat`` file (9-element solar mixture, X = 0.70, Z = 0.02).
We use ``out_of_bounds='nan'`` so that points outside the tabulated domain
are silently returned as ``NaN`` and appear as gaps in the plot.



In [ ]:
op = load_tops_opacity(mean_type="rosseland", out_of_bounds="nan")

## Temperature and Density Grids

We sweep temperature from $10^3\;\mathrm{K}$ to $2\times10^9\;\mathrm{K}$ —
slightly wider than the table domain — so the plot shows where the curves
begin and end.  Points outside the tabulated range are returned as ``NaN``
and appear as gaps.

Because the TOPS table is stored on a direct $(\log_{10}T,\,\log_{10}\rho)$
grid (unlike OPAL's $(\log_{10}T,\,\log_{10}R)$ grid), each density
curve traces a *horizontal slice* through the table; there is no
$R$-direction out-of-bounds effect as density is fixed.



In [ ]:
n_T = 800
T = np.geomspace(1e4, 2e9, n_T) * u.K

log10_rho_vals = np.array([-6.0, -5, -4.0, -3, -2.0, -1, 0.0, 1, 2.0])
rho_vals = (10.0**log10_rho_vals) * u.Unit("g cm-3")

## Evaluate $\kappa_R(T,\rho)$

For each density we broadcast the opacity call over the full temperature
array.  Out-of-domain results are returned as ``NaN`` and masked before
plotting.



In [ ]:
kappa_curves = []
for rho in rho_vals:
    kappa = op.opacity(rho * np.ones(n_T), T).to_value(u.Unit("cm2 g-1"))
    kappa_curves.append(kappa)

## Plot

Lines are colour-coded by $\log_{10}(\rho\,[\mathrm{g\,cm^{-3}}])$
using the plasma colourmap.  A shared colourbar on the right encodes the
density axis.



In [ ]:
set_plot_style()

cmap = plt.get_cmap("plasma")
norm = Normalize(vmin=log10_rho_vals.min(), vmax=log10_rho_vals.max())

fig, ax = plt.subplots(figsize=(9, 6))

for log10_rho, kappa in zip(log10_rho_vals, kappa_curves):
    color = cmap(norm(log10_rho))
    mask = np.isfinite(kappa) & (kappa > 0)
    ax.semilogy(
        T[mask].to_value(u.K),
        kappa[mask],
        color=color,
        lw=1.8,
    )

ax.set_xscale("log")
ax.set_xlabel(r"Temperature $T$ [K]")
ax.set_ylabel(r"Rosseland mean opacity $\kappa_R$ [cm$^2$ g$^{-1}$]")
ax.set_title(r"TOPS Opacity — Solar Composition ($X\approx0.70,\;Z\approx0.02$)")
ax.set_xlim(T.min().to_value(u.K), T.max().to_value(u.K))
ax.grid(True, which="both", ls="--", alpha=0.3)

# Reference line: electron-scattering approximation κ_es ≈ 0.2(1 + X) cm² g⁻¹
kappa_es = 0.2 * (1.0 + 0.70)
ax.axhline(kappa_es, color="k", ls=":", lw=1.2, alpha=0.6, label=rf"$\kappa_\mathrm{{es}}={kappa_es:.2f}$")
ax.legend(loc="upper right", fontsize=9)

# Colourbar
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(r"$\log_{10}\!\left(\rho\;[\mathrm{g\,cm^{-3}}]\right)$")

plt.tight_layout()
plt.show()

## Discussion

The curves reproduce the well-known features of Rosseland mean opacity in
stellar interiors:

- **High** $T$ **plateau** — at $T \gtrsim 10^7\;\mathrm{K}$, where
  matter is fully ionised, opacity approaches the electron-scattering value
  $\kappa_\mathrm{es} = 0.2(1+X)\;\mathrm{cm^2\,g^{-1}} \approx 0.34$
  (dashed line).
- **Kramers regime** — at intermediate temperatures the opacity rises steeply
  as $\kappa \propto \rho\,T^{-3.5}$, driven by bound-free and free-free
  absorption.  Higher-density curves are shifted upward by the $\rho$
  prefactor.
- **Iron-group peak** — the prominent bump near
  $T \sim 1$–$5 \times 10^5\;\mathrm{K}$ originates from a forest
  of bound–bound transitions of partially ionised iron and nickel.
- **Low-**$T$ **decline** — below $\sim 10^4\;\mathrm{K}$ the
  dominant absorbers (H, He) recombine, sharply reducing the opacity; these
  temperatures fall outside the TOPS table and appear as gaps.

**TOPS vs. OPAL** — both tables give consistent Rosseland mean opacities for
solar composition.  The key practical difference is the grid parameterisation:
TOPS uses a direct $(\log_{10}T,\,\log_{10}\rho)$ grid, so fixed-density
curves stay within the table for the full temperature range without the
$R$-direction out-of-bounds gaps seen in OPAL.  TOPS also covers a
wider temperature range (up to $\sim 10^9\;\mathrm{K}$) and a denser
density grid (100 points over $10^{-7}$–$10^2\;\mathrm{g\,cm^{-3}}$).



In [ ]:
sphinx_gallery_thumbnail_number = -1